In [1]:
# 1. Cài đặt bổ sung zstd để phục vụ việc giải nén Ollama
!apt-get install -y zstd

# 2. Cài đặt Ollama vào hệ thống Colab
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Cài đặt thư viện Python hỗ trợ và Giao diện Gradio
!pip install -q ollama gradio

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,137 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current 

In [2]:
import subprocess
import time

# 1. Khởi động Ollama Server chạy ngầm (Đã sửa lỗi tràn bộ đệm gây treo máy)
print("Đang khởi động Ollama Server...")
# Chuyển hướng log sang DEVNULL để giải phóng bộ nhớ đệm, không bao giờ bị đơ nữa
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Chờ 5 giây để server khởi động hoàn toàn

# 2. Tải mô hình Qwen 2.5 (3B) về Colab
print("Đang tải mô hình Qwen2.5:3b...")
!ollama pull qwen2.5:3b
print("Tải mô hình thành công!")
print("Tải mô hình thành công!")

Đang khởi động Ollama Server...
Đang tải mô hình Qwen2.5:3b...

Tải mô hình thành công!
Tải mô hình thành công!


In [3]:
import os
import json
import re
import csv
import pandas as pd
import plotly.express as px
import gradio as gr
import ollama
from google.colab import drive
from datetime import datetime

# ==========================================
# 1. KẾT NỐI GOOGLE DRIVE VÀ DATABASE
# ==========================================
drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/AI_Finance_Ollama/'
if not os.path.exists(drive_path):
    os.makedirs(drive_path)

file_path = os.path.join(drive_path, 'bank_database.json')
csv_path = os.path.join(drive_path, 'sao_ke_giao_dich.csv')

def load_db():
    if not os.path.exists(file_path):
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump({"balance": 0, "transactions": []}, f, ensure_ascii=False, indent=4)
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_db(data):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def export_to_csv(transactions):
    with open(csv_path, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(["Thời gian", "Hành động", "Số tiền (VNĐ)", "Danh mục", "Mô tả", "Số dư sau GD"])
        for tx in transactions:
            writer.writerow([tx['time'], tx['action'], tx['amount'], tx['category'], tx['description'], tx['balance_after']])
    return csv_path

# ==========================================
# 2. HỆ THỐNG AI LÕI - AI CHỈ TRÍCH XUẤT, PYTHON TÍNH TOÁN
# ==========================================
def smart_bank_agent(message, history):
    db = load_db()
    now = datetime.now()
    system_time = now.strftime("%d/%m/%Y %H:%M:%S")
    today_str = now.strftime("%d/%m/%Y")

    # PROMPT ĐÃ ĐƯỢC ÉP BUỘC KỶ LUẬT NGÔN NGỮ 100% TIẾNG VIỆT
    system_prompt = (
        "Bạn là AI trích xuất dữ liệu tài chính. YÊU CẦU TỐI THƯỢNG: TRẢ LỜI 100% BẰNG TIẾNG VIỆT.\n"
        "TUYỆT ĐỐI CẤM SỬ DỤNG TIẾNG INDONESIA (như makan, membeli...), TIẾNG ANH, HAY CHỮ HÁN.\n\n"

        "=== HƯỚNG DẪN PHÂN LOẠI HÀNH ĐỘNG (hanh_dong) ===\n"
        "- 'chi': Người dùng THÔNG BÁO VỪA TIÊU TIỀN, MUA SẮM.\n"
        "- 'thu': Người dùng THÔNG BÁO VỪA NHẬN TIỀN, CÓ LƯƠNG, ĐƯỢC THÊM, ĐƯỢC CHO.\n"
        "- 'tu_van': Người dùng XIN LỜI KHUYÊN, hỏi cách tiết kiệm, nhận xét chi tiêu.\n"
        "- 'thi_truong': Người dùng hỏi về đầu tư, giá vàng, chứng khoán.\n"
        "- 'khoi_tao': Cài đặt số dư gốc ban đầu.\n"
        "- 'tra_cuu': Xem lịch sử, sao kê.\n"
        "- 'thong_ke': Tổng hợp chi tiêu theo ngày.\n"
        "- 'xuat_file': Yêu cầu tải file csv.\n"
        "- 'xoa_du_lieu': Xóa toàn bộ dữ liệu.\n"
        "- 'khong_ro': Các câu hỏi ngoài lề.\n\n"

        "=== QUY TẮC TRÍCH XUẤT TIỀN (QUAN TRỌNG NHẤT) ===\n"
        "- BẠN BỊ CẤM TÍNH TOÁN HAY QUY ĐỔI SỐ TIỀN.\n"
        "- Trích xuất Y NGUYÊN chuỗi tiền tệ người dùng nhập vào một mảng CHUỖI.\n"
        "- Ví dụ: 'mua trà sữa 35k và ăn tối 60k' -> cac_khoan_tien: [\"35k\", \"60k\"]\n"
        "- Ví dụ: 'thêm 1 củ rưỡi' -> cac_khoan_tien: [\"1.5 củ\"]\n"
        "- Nếu người dùng nói khuyết số như 'mất lít', 'thêm củ' -> trích xuất nguyên văn [\"lít\"], [\"củ\"].\n"
        "- NẾU KHÔNG CÓ TIỀN, BẮT BUỘC ĐIỀN [\"0\"].\n\n"

        "Định dạng JSON bắt buộc:\n"
        "{\n"
        '  "hanh_dong": "<Chọn 1 hành động>",\n'
        '  "cac_khoan_tien": <MẢNG CHUỖI NHƯ HƯỚNG DẪN TRÊN>,\n'
        '  "danh_muc": "<Ăn uống, Lặt vặt, Di chuyển, Mua sắm, Hóa đơn, Giải trí, Thu nhập, Khác>",\n'
        '  "mo_ta": "<Tóm tắt lý do giao dịch hoàn toàn BẰNG TIẾNG VIỆT CÓ DẤU>",\n'
        '  "hom_nay": <true/false>\n'
        "}"
    )

    try:
        response = ollama.chat(
            model='qwen2.5:3b',
            options={'temperature': 0.0},
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': message}
            ]
        )

        raw_text = response['message']['content']
        json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)

        if json_match:
            ai_data = json.loads(json_match.group(0))
        else:
            return "❌ Lỗi: Không thể nhận diện yêu cầu. Vui lòng thử lại."

        action = ai_data.get("hanh_dong", "")

        # ==========================================
        # PYTHON DỊCH TIỀN TỆ & TÍNH TỔNG (CHÍNH XÁC 100%)
        # ==========================================
        tien_data = ai_data.get("cac_khoan_tien", ["0"])
        if not isinstance(tien_data, list):
            tien_data = [tien_data]

        amount = 0
        for val in tien_data:
            val_str = str(val).lower()
            nums = re.findall(r'\d+(?:\.\d+)?', val_str)
            base_val = float(nums[0]) if nums else 1.0

            if any(x in val_str for x in ['củ', 'triệu', 'tr', 'bánh']):
                amount += int(base_val * 1000000)
            elif any(x in val_str for x in ['lít', 'loét']):
                amount += int(base_val * 100000)
            elif any(x in val_str for x in ['k', 'cành', 'nghìn', 'ngàn']):
                amount += int(base_val * 1000)
            else:
                if nums:
                    amount += int(base_val)

        category = ai_data.get("danh_muc", "Khác")
        desc = ai_data.get("mo_ta", "")

        reply_message = ""

        if action in ["thi_truong", "tu_van"]:
            total_chi = sum(tx['amount'] for tx in db["transactions"] if tx['action'] == 'chi')
            chi_dict = {}
            for tx in db["transactions"]:
                if tx['action'] == 'chi':
                    chi_dict[tx['category']] = chi_dict.get(tx['category'], 0) + tx['amount']

            chi_breakdown = "\n".join([f"  + {k}: {v:,} VNĐ" for k, v in chi_dict.items()]) if chi_dict else "  + Chưa phát sinh khoản chi nào."

            market_prompt = (
                f"Người dùng vừa hỏi: '{message}'.\n\n"
                f"Dữ liệu ví hiện tại: Số dư {db['balance']:,} VNĐ. Tổng đã chi {total_chi:,} VNĐ.\n"
                f"Chi tiết từng danh mục đã chi tiêu:\n{chi_breakdown}\n\n"
                "Hãy đóng vai chuyên gia tài chính trả lời thẳng vào vấn đề người dùng hỏi. "
                "TUYỆT ĐỐI CHỈ SỬ DỤNG CÁC CON SỐ TRONG DANH SÁCH TRÊN KHI BẠN NHẮC ĐẾN CÁC DANH MỤC. CẤM TỰ BỊA RA HOẶC PHÓNG ĐẠI SỐ TIỀN.\n"
                "Bằng tiếng Việt. CẤM TUYỆT ĐỐI SỬ DỤNG TIẾNG TRUNG QUỐC."
            )

            market_response = ollama.chat(
                model='qwen2.5:3b',
                options={'temperature': 0.7},
                messages=[{'role': 'user', 'content': market_prompt}]
            )
            ai_advice = re.sub(r'[\u4e00-\u9fff]+', '', market_response['message']['content']).strip()
            return f"💡 **CHUYÊN GIA TÀI CHÍNH AI TRẢ LỜI:**\n\n{ai_advice}"

        elif action == "khong_ro": return "⚠️ Vui lòng cung cấp giao dịch rõ ràng."

        elif action == "xoa_du_lieu":
            db["balance"] = 0
            db["transactions"] = []
            save_db(db)
            return "🗑️ **ĐÃ XÓA DỮ LIỆU**\n\nToàn bộ lịch sử giao dịch và số dư đã được làm mới về 0 VNĐ."

        elif action == "thong_ke":
            if not db["transactions"]: return "⚠️ Chưa có dữ liệu giao dịch."
            daily_stats = {}
            for tx in db["transactions"]:
                if tx['action'] == 'chi':
                    date_str = tx['time'].split(' ')[0]
                    if date_str not in daily_stats: daily_stats[date_str] = {'tong_chi': 0, 'tien_an': 0}
                    daily_stats[date_str]['tong_chi'] += tx['amount']
                    if tx['category'] == 'Ăn uống': daily_stats[date_str]['tien_an'] += tx['amount']

            if not daily_stats: return "📝 Chưa có khoản chi tiêu nào."
            report = "📈 **BÁO CÁO CHI TIÊU THEO NGÀY**\n\n"
            for date_str, stats in sorted(daily_stats.items(), reverse=True):
                report += f"📅 **{date_str}**\n  • 🍔 Riêng tiền ăn: **-{stats['tien_an']:,} VNĐ**\n  • 📉 Tổng chi: **-{stats['tong_chi']:,} VNĐ**\n  ---\n"
            reply_message = report.strip()

        elif action == "khoi_tao":
            db["balance"] = amount
            reply_message = f"🔵 **THIẾT LẬP TÀI KHOẢN**\n\n• Ngày: {system_time}\n• Số dư: **{amount:,} VNĐ**"

        elif action == "thu":
            db["balance"] += amount
            reply_message = f"🟢 **BIẾN ĐỘNG SỐ DƯ (THU)**\n\n• Số tiền: **+{amount:,} VNĐ**\n• Danh mục: {category}\n• Lời nhắn: {desc}\n\n💳 Số dư: **{db['balance']:,} VNĐ**"

        elif action == "chi":
            db["balance"] -= amount
            reply_message = f"🔴 **BIẾN ĐỘNG SỐ DƯ (CHI)**\n\n• Số tiền: **-{amount:,} VNĐ**\n• Danh mục: {category}\n• Lời nhắn: {desc}\n\n💳 Số dư: **{db['balance']:,} VNĐ**"

        elif action == "tra_cuu":
            is_today = ai_data.get("hom_nay", False)
            if "hôm nay" in message.lower(): is_today = True
            filtered_tx = db["transactions"]
            if is_today: filtered_tx = [tx for tx in filtered_tx if tx['time'].startswith(today_str)]

            limit = 10
            nums = re.findall(r'\d+', message)
            if nums and ("lần" in message.lower() or "giao dịch" in message.lower()): limit = int(nums[0])

            filtered_tx = filtered_tx[-limit:]
            title_str = "HÔM NAY" if is_today else "GẦN NHẤT"
            history_str = f"\n\n📋 **SAO KÊ {len(filtered_tx)} GIAO DỊCH {title_str}:**\n"

            if not filtered_tx:
                history_str += "*Chưa có giao dịch nào phù hợp.*"
            else:
                for tx in filtered_tx:
                    icon, sign = ("🟢", "+") if tx['action'] == 'thu' else ("🔴", "-")
                    if tx['action'] == 'khoi_tao':
                        history_str += f"• 🔵 [{tx['time']}] **Khởi tạo số dư**: {tx['amount']:,} VNĐ\n"
                    else:
                        history_str += f"• {icon} [{tx['time']}] **{tx['category']}**: {sign}{tx['amount']:,} VNĐ ({tx['description']})\n"

            reply_message = f"🏦 **TỔNG KẾT TÀI KHOẢN**\n\n💳 Số dư: **{db['balance']:,} VNĐ**" + history_str

        elif action == "xuat_file":
            if not db["transactions"]: return "⚠️ Hệ thống chưa có dữ liệu giao dịch nào để xuất file."
            export_to_csv(db["transactions"])
            reply_message = f"📊 **XUẤT DỮ LIỆU THÀNH CÔNG**\n\nFile sao kê (CSV) đã được tạo tự động.\n📍 **Vị trí lưu:** Google Drive > `AI_Finance_Ollama/sao_ke_giao_dich.csv`"

        else:
            return "⚠️ Hệ thống không hiểu ý bạn, vui lòng nhập lại."

        if action in ["thu", "chi", "khoi_tao"] and amount > 0:
            db["transactions"].append({
                "time": system_time, "action": action, "amount": amount,
                "category": category, "description": desc if desc else message, "balance_after": db["balance"]
            })
            save_db(db)

        return reply_message

    except Exception as e:
        return f"⚠️ Lỗi xử lý hệ thống: (Chi tiết: {str(e)})"


# ==========================================
# 3. HÀM CẬP NHẬT GIAO DIỆN DASHBOARD
# ==========================================
def refresh_dashboard():
    db = load_db()
    balance = f"{db['balance']:,} VNĐ"

    txs = db["transactions"]
    if not txs:
        return balance, "0 VNĐ", "0 VNĐ", None, pd.DataFrame(columns=["Thời gian", "Hành động", "Số tiền (VNĐ)", "Danh mục", "Mô tả", "Số dư"])

    df = pd.DataFrame(txs)
    thu = df[df['action'] == 'thu']['amount'].sum()
    chi = df[df['action'] == 'chi']['amount'].sum()

    df_chi = df[df['action'] == 'chi']
    if not df_chi.empty:
        cat_sum = df_chi.groupby('category')['amount'].sum().reset_index()
        fig = px.pie(
            cat_sum,
            values='amount',
            names='category',
            title="CƠ CẤU CHI TIÊU",
            hole=0.45,
            template="plotly_dark",
            color_discrete_sequence=px.colors.qualitative.Pastel
        )

        fig.update_layout(
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(0,0,0,0)",
            font=dict(family="Arial", size=14, color="white"),
            title_x=0.5,
            title_font_size=20,
            margin=dict(t=60, b=20, l=20, r=20)
        )

        fig.update_traces(
            textposition='inside',
            textinfo='percent+label',
            marker=dict(line=dict(color='#000000', width=1))
        )
    else:
        fig = None

    display_df = df[["time", "action", "amount", "category", "description", "balance_after"]].copy()
    display_df.columns = ["Thời gian", "Hành động", "Số tiền (VNĐ)", "Danh mục", "Mô tả", "Số dư sau GD"]

    return balance, f"+{thu:,} VNĐ", f"-{chi:,} VNĐ", fig, display_df

def respond(message, chat_history):
    bot_reply = smart_bank_agent(message, chat_history)
    chat_history.append((message, bot_reply))
    return "", chat_history

# ==========================================
# 4. GIAO DIỆN WEB DẠNG ĐA TAB (BLOCKS)
# ==========================================
with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("<h1 style='text-align: center;'>📊 Hệ Thống Quản Lý Tài Chính & AI Agent</h1>")
    gr.Markdown("<p style='text-align: center;'>Đồ án môn học: Tích hợp Trực quan hóa Dữ liệu và AI Agent phân tích Thị trường.</p>")

    with gr.Tabs():
        with gr.TabItem("🏠 Trang Chủ (Dashboard)"):
            btn_refresh = gr.Button("🔄 Cập Nhật Dữ Liệu Bảng Giám Sát", variant="primary")
            with gr.Row():
                val_balance = gr.Textbox(label="💳 Số Dư Hiện Tại", interactive=False, text_align="center")
                val_thu = gr.Textbox(label="📈 Tổng Thu", interactive=False, text_align="center")
                val_chi = gr.Textbox(label="📉 Tổng Chi", interactive=False, text_align="center")
            with gr.Row():
                chart = gr.Plot(label="Trực quan hóa Dữ liệu")

        with gr.TabItem("🤖 Trợ Lý AI (Agent)"):
            gr.Markdown("Tích hợp AI NLP xử lý: **Giao dịch tài chính**, **Thống kê**, và **Phân tích Thị trường**")
            chatbot = gr.Chatbot(height=400)
            msg = gr.Textbox(label="Nhập câu lệnh cho AI Agent...")
            clear = gr.Button("🗑️ Xóa hội thoại")

        with gr.TabItem("📋 Danh sách Giao Dịch"):
            gr.Markdown("Lịch sử biến động số dư được đồng bộ trực tiếp từ AI Agent.")
            table = gr.Dataframe(headers=["Thời gian", "Hành động", "Số tiền (VNĐ)", "Danh mục", "Mô tả", "Số dư sau GD"])

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)
    btn_refresh.click(refresh_dashboard, inputs=[], outputs=[val_balance, val_thu, val_chi, chart, table])
    demo.load(refresh_dashboard, inputs=[], outputs=[val_balance, val_thu, val_chi, chart, table])

if __name__ == "__main__":
    demo.launch(share=True)

Mounted at /content/drive


/tmp/ipykernel_2535/4075222002.py:301: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
/tmp/ipykernel_2535/4075222002.py:317: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_2535/4075222002.py:317: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c5643b93472959c032.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
